In [65]:
from typing import List
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

import cvxpy as cp

from Components.RecedingHorizon import RecedingHorizon
from Components.System import SimpleSquareExample
from Components.Model import SquareCoarseModel
from Components.Controllers import LTI_LQRc_RHG

In [55]:
# Reload scripts when executed
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
parameter_file = "Components/Setup/two_cells_aquifer.json"

In [57]:
# Initialize Model
model = SquareCoarseModel(parameter_file)

In [58]:
model.A.toarray(), model.B_v[0].toarray()

(array([[0.9796, 0.0204],
        [0.0204, 0.9796]]),
 array([[2.e-05],
        [0.e+00]]))

In [59]:
# Initialize Controller
controller = LTI_LQRc_RHG(
    parameter_file,
    model.A.toarray(), [B.toarray() for B in model.B_v], model.C,
    model.H_x, model.h_x, model.H_u_v, model.h_u_v,
    Q_v = None, R_v = None, M_v = [G.toarray() for G in model.Gamma_v], L_v = [Xi for Xi in model.Xi_v],
    solver='fbrs'
)

In [60]:
controller.A_tilde.toarray(), controller.B_v_tilde[0].toarray()*1e5, controller.B_v_tilde[1].toarray()*1e5

(array([[1.        , 0.        ],
        [0.        , 1.        ],
        [0.9796    , 0.0204    ],
        [0.0204    , 0.9796    ],
        [0.96003232, 0.03996768],
        [0.03996768, 0.96003232],
        [0.941263  , 0.058737  ],
        [0.058737  , 0.941263  ]]),
 array([[0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [2.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [1.9592    , 2.        , 0.        ],
        [0.0408    , 0.        , 0.        ],
        [1.92006464, 1.9592    , 2.        ],
        [0.07993536, 0.0408    , 0.        ]]),
 array([[0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [2.        , 0.        , 0.        ],
        [0.0408    , 0.        , 0.        ],
        [1.9592    , 2.        , 0.        ],
        [0.07993536, 0.0408    , 0.        ],
        [1.92006464, 1.9592    , 2

In [61]:
controller.F.toarray(), controller.f_x.toarray(), controller.f.toarray()

(array([[0.      , 0.02    , 0.019592, 0.      , 0.      , 0.      ],
        [0.02    , 0.      , 0.02    , 0.      , 0.      , 0.      ],
        [0.019592, 0.02    , 0.      , 0.000408, 0.      , 0.      ],
        [0.      , 0.      , 0.      , 0.      , 0.02    , 0.019592],
        [0.      , 0.      , 0.      , 0.02    , 0.      , 0.02    ],
        [0.000408, 0.      , 0.      , 0.019592, 0.02    , 0.      ]]),
 array([[1000.     ,    0.     ],
        [ 979.6    ,   20.4    ],
        [ 960.03232,   39.96768],
        [   0.     , 1000.     ],
        [  20.4    ,  979.6    ],
        [  39.96768,  960.03232]]),
 array([[-8000.],
        [-8000.],
        [-8000.],
        [-8000.],
        [-8000.],
        [-8000.]]))

In [73]:
(0.5*(controller.F.T + controller.F)).toarray()

array([[0.      , 0.02    , 0.019592, 0.      , 0.      , 0.000204],
       [0.02    , 0.      , 0.02    , 0.      , 0.      , 0.      ],
       [0.019592, 0.02    , 0.      , 0.000204, 0.      , 0.      ],
       [0.      , 0.      , 0.000204, 0.      , 0.02    , 0.019592],
       [0.      , 0.      , 0.      , 0.02    , 0.      , 0.02    ],
       [0.000204, 0.      , 0.      , 0.019592, 0.02    , 0.      ]])

In [77]:
controller.H = sp.csr_array(np.zeros_like(controller.H.toarray()))
controller.h_x_0 = sp.csr_array(np.zeros_like(controller.h_x_0.toarray()))
controller.h = sp.csr_array(np.zeros_like(controller.h.toarray()))

In [139]:
x_0 = 10. * np.ones((2,1))
u_opt = controller.step(x_0)
u_opt

array([[-49494.94949495, -49494.94949495],
       [-50505.05050505, -50505.05050505],
       [-50505.05050505, -50505.05050505]])

In [133]:
(controller.F.T + controller.F).toarray()

array([[0.      , 0.04    , 0.039184, 0.      , 0.      , 0.000408],
       [0.04    , 0.      , 0.04    , 0.      , 0.      , 0.      ],
       [0.039184, 0.04    , 0.      , 0.000408, 0.      , 0.      ],
       [0.      , 0.      , 0.000408, 0.      , 0.04    , 0.039184],
       [0.      , 0.      , 0.      , 0.04    , 0.      , 0.04    ],
       [0.000408, 0.      , 0.      , 0.039184, 0.04    , 0.      ]])

In [ ]:
x = cp.Variable(6)
prob = cp.Problem(0.5*cp.Minimize(cp.quad_form(x, (controller.F.T + controller.F).toarray()) + (controller.f_x @ x_0 + controller.f).T @ x))
prob.solve()

DCPError: Problem does not follow DCP rules. Specifically:
The objective is not DCP. Its following subexpressions are not:
QuadForm(var40, [[0.00 0.04 ... 0.00 0.00]
 [0.04 0.00 ... 0.00 0.00]
 ...
 [0.00 0.00 ... 0.00 0.04]
 [0.00 0.00 ... 0.04 0.00]])

In [76]:
np.linalg.eigvals((0.5*(controller.F.T + controller.F)).toarray())

array([ 0.03986415,  0.03959339, -0.019388  , -0.019796  , -0.02020539,
       -0.02006815])

In [102]:
F = controller.F.copy()
F[2,3] = 0
F[-1,0] = 0
np.linalg.eigvals(F.toarray())

array([ 0.03972862, -0.019592  , -0.02013662,  0.03972862, -0.019592  ,
       -0.02013662])

In [91]:
test = np.zeros((3,3))
test[0,1] = test[1,0] = test[1,2] = test[2,1] = -5.
test[0,2] = test[2,0] = -4.4
np.linalg.eigvals(test)

array([-9.60540343,  4.4       ,  5.20540343])

In [92]:
mu_h = model.A[0,1]
mu_q = model.B_v[0][0,0]
rho = model.Gamma_v[0][0,0]

(1-mu_h)*mu_q*rho, 0.5*(-mu_q*rho*np.sqrt((1-mu_h)**2 + 8) - (1-mu_h)*mu_q*rho), 0.5*(mu_q*rho*np.sqrt((1-mu_h)**2 + 8) - (1-mu_h)*mu_q*rho)

(np.float64(1.9592e-05),
 np.float64(-3.972861792760533e-05),
 np.float64(2.013661792760533e-05))

In [94]:
np.linalg.eigvals(controller.F.toarray()[:3,:3])*1e-3

array([ 3.97286179e-05, -1.95920000e-05, -2.01366179e-05])

By definition M_{11} and M_{22} are not positive semi-definite (nor negative)

F is not positive semi-definite either and the problem is therefore not convex

In [154]:
import gurobipy as gp

m = gp.Model("matrix1")
u = m.addMVar(shape=6, vtype=gp.GRB.CONTINUOUS, name="U")
m.setObjective(u.T @ (0.25*controller.F.T + 0.25*controller.F).toarray() @ u + -(controller.f_x @ x_0 + controller.f).T @ u, gp.GRB.MINIMIZE)
m.optimize()

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[x86] - Darwin 24.4.0 24E263)

CPU model: Intel(R) Core(TM) i5-1038NG7 CPU @ 2.00GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 0 rows, 6 columns and 0 nonzeros
Model fingerprint: 0x6ca92a15
Model has 8 quadratic objective terms
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  Objective range  [2e+03, 2e+03]
  QObjective range [4e-04, 4e-02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [0e+00, 0e+00]

Continuous model is non-convex -- solving as a MIP

Found heuristic solution: objective 0.0000000
Found heuristic solution: objective -1199.998804
Presolve time: 0.02s
Presolved: 17 rows, 16 columns, 33 nonzeros
Presolved model has 8 bilinear constraint(s)
         in product terms.
         Presolve was not able to compute smaller bounds for these variables.
         Consider bounding these variables or reformulating the model.

Variable types: 16 continuous, 

In [155]:
u.X, m.ObjVal

(array([7.53761535e+15, 0.00000000e+00, 0.00000000e+00, 7.53761535e+15,
        0.00000000e+00, 0.00000000e+00]),
 -3.015046141143175e+19)

In [108]:
u_test = -1000*np.ones((6,1))
u_test.T @ controller.F @ u_test + (controller.f_x @ x_0 + controller.f).T @ u_test

array([[-11760816.]])

In [145]:
u_opt_f = u_opt.T.reshape((6,1))
u_opt_f.T @ controller.F @ u_opt_f + (controller.f_x @ x_0 + controller.f).T @ u_opt_f

array([[-39995.91878378]])

In [ ]:
(-u.X).T @ controller.F @ (-u.X) + (controller.f_x @ x_0 + controller.f).T @ (-u.X)

array([-6.8e+10])

In [158]:
u_analytic = np.linalg.inv((controller.F).toarray()) @ (-(controller.f_x @ x_0 + controller.f))
u_analytic

array([[-49484.74439281],
       [-50515.25560719],
       [-50515.25560719],
       [-49484.74439281],
       [-50515.25560719],
       [-50515.25560719]])

In [163]:
0.5*(controller.F+controller.F.T) @ u_analytic + controller.f_x @ x_0 + controller.f

array([[-10.30511214],
       [  0.        ],
       [ 10.09488786],
       [-10.30511214],
       [  0.        ],
       [ 10.09488786]])

F is not symmetric and cannot be integrated.